# Demo Pipeline: рекомендательная система маршрутов

Этот ноутбук показывает полный MVP-пайплайн рекомендательной системы для каталога hiking/tourism маршрутов на полностью синтетических данных.

В проекте нет клиентских данных, production-кода, proprietary database schema, внутренних бизнес-метрик или customer-specific business logic.

## 1. Загрузка synthetic dataset

Стартовая точка пайплайна — воспроизводимые CSV-файлы в `data/`. Первая ячейка также добавляет локальный `src/` в `sys.path`, чтобы ноутбук запускался из Jupyter без обязательного `pip install -e .`.

In [1]:
from pathlib import Path
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from hiking_recommender.data_loader import load_demo_dataset

dataset = load_demo_dataset(project_root / "data")

{
    "users": len(dataset.users),
    "routes": len(dataset.routes),
    "train_interactions": len(dataset.train_interactions),
    "test_interactions": len(dataset.test_interactions),
}

{'users': 200,
 'routes': 120,
 'train_interactions': 4174,
 'test_interactions': 1042}

**Резюме этапа:** загружены `200` synthetic users, `120` synthetic routes, `4174` train interactions и `1042` test interactions. Этого достаточно для компактной demo-проверки warm-user, cold-start/fallback и offline evaluation без исследовательской сложности.

In [2]:
dataset.routes.head()

,length_km,popularity,region,season,route_tags,duration_hours,elevation_gain_m,route_id,difficulty
0,2.5,0.480,east,winter,family|forest|waterfall,1.0,210,route_001,easy
1,4.8,0.186,central,summer,forest|wildlife,1.5,99,route_002,easy
2,12.2,0.731,central,winter,family|forest|lake,4.7,279,route_003,moderate
3,4.5,0.686,north,autumn,waterfall|wildlife,1.6,100,route_004,easy
4,7.9,0.263,north,summer,forest|viewpoint|waterfall,4.2,774,route_005,moderate


**Резюме по данным маршрутов:** route catalog содержит только generic synthetic признаки: `route_id`, `region`, `length_km`, `duration_hours`, `elevation_gain_m`, `difficulty`, `season`, `route_tags`, `popularity`. В примере нет реальных названий маршрутов или регионов; используются публичные synthetic IDs вида `route_001`.

## 2. Feature engineering

Feature layer отделён от загрузки данных. Здесь строятся route-level признаки и user-item matrix по implicit feedback weights.

In [3]:
from hiking_recommender.features import build_features, build_seen_routes

features = build_features(dataset)
route_features = features["route_features"]
user_item_matrix = features["user_item_matrix"]

{
    "route_features_shape": route_features.shape,
    "user_item_matrix_shape": user_item_matrix.shape,
}

{'route_features_shape': (120, 19), 'user_item_matrix_shape': (200, 120)}

**Резюме этапа:** получена feature matrix маршрутов размера `(120, 19)` и user-item matrix размера `(200, 120)`. Это подтверждает, что все users/routes представлены в train контуре, а retrieval-модели могут работать поверх общей матрицы взаимодействий.

## 3. Retrieval sources: запуск моделей по отдельности

Сначала запускаем три модели независимо друг от друга. Это важно для baseline-first подхода: до hybrid merge нужно понимать, что даёт каждый retrieval source сам по себе.

Источники кандидатов:

- `PopularityRecommender` — популярные маршруты и fallback;
- `CollaborativeRecommender` — item-item similarity по implicit feedback;
- `ContentBasedRecommender` — похожесть маршрутов по признакам.

In [4]:
from hiking_recommender.baseline import PopularityRecommender
from hiking_recommender.collaborative import CollaborativeRecommender
from hiking_recommender.content_based import ContentBasedRecommender

popularity = PopularityRecommender().fit(dataset)
collaborative = CollaborativeRecommender().fit(dataset)
content = ContentBasedRecommender().fit(dataset)

user_id = "user_001"
region = "north"
candidate_limit = 30

popularity_candidates = popularity.recommend(user_id, top_k=candidate_limit, region=region)
collaborative_candidates = collaborative.recommend(user_id, top_k=candidate_limit, region=region)
content_candidates = content.recommend(user_id, top_k=candidate_limit, region=region)

{
    "popularity": len(popularity_candidates),
    "collaborative": len(collaborative_candidates),
    "content": len(content_candidates),
}

{'popularity': 18, 'collaborative': 18, 'content': 18}

**Резюме этапа:** для `user_001` и региона `north` каждый retrieval source вернул по `18` кандидатов. Это значит, что region filter не обнуляет выдачу, а все три источника можно корректно сравнивать и затем объединять.

## 4. Сравнение отдельных моделей

Считаем offline top-K metrics отдельно для popularity, collaborative и content моделей. Это показывает, какой источник сильнее до merger и business rules.

**Как читать метрики для работодателя или клиента:**

| Метрика | Что показывает | Как использовать в разговоре |
|---|---|---|
| `precision@10` | Какая доля top-10 рекомендаций попала в held-out interactions. | Насколько точна выдача в первых позициях. |
| `recall@10` | Какую долю релевантных held-out маршрутов модель нашла. | Насколько хорошо модель не пропускает интересные варианты. |
| `MAP@10` | Учитывает, насколько рано появляются правильные рекомендации. | Качество порядка выдачи, а не только факт попадания. |
| `NDCG@10` | Награждает релевантные маршруты выше в списке. | Удобно сравнивать ranking quality между моделями. |
| `coverage@10` | Какую часть каталога модель вообще показывает пользователям. | Видно, не зациклилась ли модель на малом наборе items. |
| `novelty@10` | Насколько выдача уходит от самых популярных маршрутов. | Показывает контроль popularity bias. |
| `diversity@10` | Насколько маршруты внутри списков отличаются по metadata. | Показывает ширину выбора для пользователя. |

Эти метрики — offline validation на synthetic data. Их нельзя трактовать как прогноз CTR, revenue или production uplift.

In [5]:
import pandas as pd

from hiking_recommender.evaluation import build_relevance, evaluate_recommendations

cutoff = 10
user_ids = sorted(build_relevance(dataset.test_interactions))
all_route_ids = list(dataset.routes["route_id"].astype(str))


def route_ids(candidates):
    return [candidate.route_id for candidate in candidates]


single_model_recommendations = {
    "popularity": {
        current_user_id: route_ids(popularity.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
    "collaborative": {
        current_user_id: route_ids(collaborative.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
    "content": {
        current_user_id: route_ids(content.recommend(current_user_id, top_k=cutoff))
        for current_user_id in user_ids
    },
}

single_model_metrics = pd.DataFrame(
    [
        {"model": model_name, **evaluate_recommendations(recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff, train_interactions=dataset.train_interactions, routes=dataset.routes)}
        for model_name, recommendations in single_model_recommendations.items()
    ]
).sort_values("model")

metric_columns = [column for column in single_model_metrics.columns if column != "model"]
single_model_metrics_table = single_model_metrics.copy()
single_model_metrics_table[metric_columns] = single_model_metrics_table[metric_columns].round(4)

single_model_metrics_table

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10,novelty@10,diversity@10
1,collaborative,0.0790,0.1484,0.0572,0.1249,0.9167,0.5545,0.6684
2,content,0.1035,0.1898,0.0740,0.1584,1.0000,0.5761,0.5120
0,popularity,0.0735,0.1375,0.0498,0.1130,0.1833,0.5379,0.6890


**Как читать таблицу отдельных моделей:**

| Что смотрим | Лучший вариант | Что это значит |
|---|---|---|
| Ranking quality | `content` | Лучшие `precision@10`, `recall@10`, `MAP@10` и `NDCG@10` среди одиночных источников. |
| Catalog reach | `content` | `coverage@10=1.0000`: модель покрывает весь synthetic catalog в выдачах по пользователям. |
| Novelty | `content` | `novelty@10=0.5761`: меньше bias в сторону самых популярных маршрутов. |
| Diversity | `popularity` | `diversity@10=0.6890`, но при низком `coverage@10=0.1833`; это не делает popularity лучшей моделью. |

**Практический вывод:** `popularity` нужен как простой fallback, `collaborative` добавляет behavioral signal, а `content` сейчас самый сильный одиночный retrieval source для этого synthetic dataset.

## 5. Candidate merger: hybrid retrieval

Теперь объединяем три источника. Merger работает по `route_id`, удаляет дубликаты и суммирует weighted reciprocal-rank contribution. Кандидаты, найденные несколькими источниками, получают boost.

In [6]:
from hiking_recommender.merger import merge_candidates

merged = merge_candidates(
    [collaborative_candidates, content_candidates, popularity_candidates],
    top_k=candidate_limit,
)

pd.DataFrame(
    [
        {
            "route_id": candidate.route_id,
            "rank": candidate.final_rank,
            "score": round(candidate.merged_score, 5),
            "sources": ",".join(candidate.sources),
            "n_sources": candidate.n_sources,
        }
        for candidate in merged[:10]
    ]
)

,route_id,rank,score,sources,n_sources
0,route_095,1,0.03770,"collaborative,content,popularity",3
1,route_109,2,0.03595,"collaborative,content,popularity",3
2,route_102,3,0.03590,"collaborative,content,popularity",3
3,route_080,4,0.03536,"collaborative,content,popularity",3
4,route_094,5,0.03508,"collaborative,content,popularity",3
5,route_054,6,0.03394,"collaborative,content,popularity",3
6,route_010,7,0.03362,"collaborative,content,popularity",3
7,route_030,8,0.03338,"collaborative,content,popularity",3
8,route_068,9,0.03303,"collaborative,content,popularity",3
9,route_075,10,0.03296,"collaborative,content,popularity",3


**Анализ merger на примере пользователя:** top-10 после merge состоит из маршрутов, найденных всеми тремя источниками: `collaborative`, `content`, `popularity`. Это демонстрирует ожидаемый hybrid-паттерн: пересечение независимых сигналов повышает confidence, а итоговый список остаётся deduplicated.

In [7]:
hybrid_recommendations = {
    current_user_id: route_ids(
        merge_candidates(
            [
                collaborative.recommend(current_user_id, top_k=candidate_limit),
                content.recommend(current_user_id, top_k=candidate_limit),
                popularity.recommend(current_user_id, top_k=candidate_limit),
            ],
            top_k=cutoff,
        )
    )
    for current_user_id in user_ids
}

hybrid_metrics = pd.DataFrame(
    [
        {"model": "hybrid", **evaluate_recommendations(hybrid_recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff, train_interactions=dataset.train_interactions, routes=dataset.routes)}
    ]
)

pd.concat([single_model_metrics, hybrid_metrics], ignore_index=True).sort_values("model")

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10,novelty@10,diversity@10
0,collaborative,0.0790,0.148393,0.057212,0.124890,0.916667,0.554514,0.668403
1,content,0.1035,0.189756,0.073973,0.158405,1.000000,0.576107,0.511998
3,hybrid,0.1055,0.193173,0.074078,0.160452,0.950000,0.554908,0.604314
2,popularity,0.0735,0.137536,0.049824,0.113029,0.183333,0.537947,0.688980


**Что изменилось после hybrid merger:**

| Метрика | Было у лучшего single source | Стало у `hybrid` | Интерпретация |
|---|---:|---:|---|
| `precision@10` | `0.1035` | `0.1055` | Небольшой прирост точности. |
| `recall@10` | `0.1898` | `0.1932` | Hybrid находит чуть больше held-out маршрутов. |
| `ndcg@10` | `0.1584` | `0.1605` | Ранжирование стало немного лучше. |
| `coverage@10` | `1.0000` | `0.9500` | Coverage снизился: merger выбирает более согласованные candidates. |
| `diversity@10` | `0.5120` у content | `0.6043` | Список стал разнообразнее, чем у content-only. |

**Практический вывод:** hybrid merger даёт небольшой, но устойчивый прирост ranking quality и сохраняет высокий catalog reach. Это хороший MVP-результат: несколько независимых retrieval sources подтверждают друг друга, а итоговая выдача остаётся deduplicated.

## 6. Business rules

Business rules применяются после retrieval и merge. В этом demo используются hard filters по региону, максимальной сложности, исключение уже виденных маршрутов и fallback fill. Fallback не должен обходить hard filters.

In [8]:
from hiking_recommender.business_rules import apply_business_rules

seen_routes = build_seen_routes(dataset.train_interactions)
fallback = merge_candidates([popularity_candidates], top_k=candidate_limit)

final_recommendations = apply_business_rules(
    merged,
    routes=dataset.routes,
    top_k=10,
    region=region,
    max_difficulty="moderate",
    seen_route_ids=seen_routes.get(user_id, set()),
    fallback_candidates=fallback,
)

route_lookup = dataset.routes.set_index("route_id")
pd.DataFrame(
    [
        {
            "route_id": candidate.route_id,
            "rank": candidate.final_rank,
            "score": round(candidate.merged_score, 5),
            "region": route_lookup.loc[candidate.route_id, "region"],
            "difficulty": route_lookup.loc[candidate.route_id, "difficulty"],
            "sources": ",".join(candidate.sources),
        }
        for candidate in final_recommendations
    ]
)

,route_id,rank,score,region,difficulty,sources
0,route_095,1,0.03770,north,easy,"collaborative,content,popularity"
1,route_109,2,0.03595,north,easy,"collaborative,content,popularity"
2,route_102,3,0.03590,north,easy,"collaborative,content,popularity"
3,route_080,4,0.03536,north,moderate,"collaborative,content,popularity"
4,route_094,5,0.03508,north,easy,"collaborative,content,popularity"
5,route_054,6,0.03394,north,moderate,"collaborative,content,popularity"
6,route_010,7,0.03362,north,moderate,"collaborative,content,popularity"
7,route_030,8,0.03338,north,moderate,"collaborative,content,popularity"
8,route_075,9,0.03296,north,moderate,"collaborative,content,popularity"
9,route_018,10,0.03291,north,easy,"collaborative,content,popularity"


**Анализ business rules на примере пользователя:** финальная выдача содержит `10` маршрутов только из региона `north`; сложности — `easy` и `moderate`, то есть фильтр `max_difficulty='moderate'` соблюдён. Ранги пересчитаны в диапазоне `1..10`, дубликатов `route_id` нет.

In [9]:
hybrid_with_rules_recommendations = {
    current_user_id: route_ids(
        apply_business_rules(
            merge_candidates(
                [
                    collaborative.recommend(current_user_id, top_k=candidate_limit),
                    content.recommend(current_user_id, top_k=candidate_limit),
                    popularity.recommend(current_user_id, top_k=candidate_limit),
                ],
                top_k=candidate_limit,
            ),
            routes=dataset.routes,
            top_k=cutoff,
            max_difficulty="moderate",
            seen_route_ids=seen_routes.get(current_user_id, set()),
            fallback_candidates=merge_candidates(
                [popularity.recommend(current_user_id, top_k=candidate_limit)],
                top_k=candidate_limit,
            ),
        )
    )
    for current_user_id in user_ids
}

hybrid_with_rules_metrics = pd.DataFrame(
    [
        {"model": "hybrid_with_rules", **evaluate_recommendations(hybrid_with_rules_recommendations, dataset.test_interactions, all_route_ids, cutoff=cutoff, train_interactions=dataset.train_interactions, routes=dataset.routes)}
    ]
)

full_metrics = pd.concat(
    [single_model_metrics, hybrid_metrics, hybrid_with_rules_metrics],
    ignore_index=True,
).sort_values("model")

full_metrics

,model,precision@10,recall@10,map@10,ndcg@10,coverage@10,novelty@10,diversity@10
0,collaborative,0.0790,0.148393,0.057212,0.124890,0.916667,0.554514,0.668403
1,content,0.1035,0.189756,0.073973,0.158405,1.000000,0.576107,0.511998
3,hybrid,0.1055,0.193173,0.074078,0.160452,0.950000,0.554908,0.604314
4,hybrid_with_rules,0.1000,0.185238,0.074095,0.156739,0.800000,0.552454,0.581814
2,popularity,0.0735,0.137536,0.049824,0.113029,0.183333,0.537947,0.688980


**Что изменилось после `apply_business_rules`:**

| Метрика | До правил: `hybrid` | После правил | Почему изменилось |
|---|---:|---:|---|
| `precision@10` | `0.1055` | `0.1000` | Hard filters убирают часть релевантных, но неподходящих candidates. |
| `recall@10` | `0.1932` | `0.1852` | Меньше доступное пространство рекомендаций. |
| `ndcg@10` | `0.1605` | `0.1567` | Ранжирование слегка просело после продуктовых ограничений. |
| `coverage@10` | `0.9500` | `0.8000` | Фильтры по difficulty/seen routes сужают catalog reach. |
| `novelty@10` | `0.5549` | `0.5525` | Popularity bias почти не изменился. |
| `diversity@10` | `0.6043` | `0.5818` | Разнообразие немного снизилось из-за hard constraints. |

**Практический вывод:** это ожидаемый product trade-off. Offline ranking немного хуже, зато финальная выдача соблюдает ограничения пользователя: `max_difficulty`, region/seen filters и fallback без обхода hard filters.

## 7. API payload example

Тот же pipeline доступен через FastAPI endpoint `POST /recommendations`. Ноутбук не поднимает сервер, а показывает payload, который соответствует публичному API contract.

In [10]:
api_payload = {
    "user_id": user_id,
    "region": region,
    "top_k": 5,
    "max_difficulty": "moderate",
}

api_payload

{'user_id': 'user_001',
 'region': 'north',
 'top_k': 5,
 'max_difficulty': 'moderate'}

**Резюме этапа:** API-запрос минимальный и продуктово понятный: `user_id`, optional `region`, `top_k`, optional `max_difficulty`. Такой контракт удобно показывать в README, curl-примере и smoke-тестах API.